In [ ]:
import pandas as pd
import io

In [ ]:
# Raw logger CSV URLs pulled directly from GitHub — one dict per deployment year
# Each year's data arrived as multiple monthly files from the HOBO device
csv_files_2022 = {
    "2022_1" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2022/1st_8-2-22_to_9-26-22.csv",
    "2022_2" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2022/2nd_Log9-26-22_to_11-20-22.csv"
}

csv_files_21023 = {
    "2023_1" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2023/Start_8-17-23_to_9-3-23.csv",
    "2023_2" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2023/Start_9-3-23_to_9-17-23.csv",
    "2023_3" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2023/Start_9-17-23_to_10-30-23.csv"
}

csv_files_2024 = {
    "2024_1" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2024/4-16-24_to_6-10-24.csv",
    "2024_2" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2024/6-10-24_to_7-12-24.csv",
    "2024_3" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2024/7-12-24_to_8-16-24.csv",
    "2024_4" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2024/8-17-24_to_12-1-24_Local1.csv",

}

csv_files_2025 = {
    "2025_1" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2025/6-13-25_at_EMF.csv",
    "2025_2" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2025/Mid-Pond_7-3-25_to_9-16-25.csv",
    "2025_3" : "https://raw.githubusercontent.com/GreenVaultJoe/capstone_Chrysler_Pond/refs/heads/main/data/raw_data/2025/Mid-Pond_9-16-25_to_11-22-25.csv"
}

# Merge all year dicts into one
csv_files = csv_files_2022 | csv_files_21023 | csv_files_2024 | csv_files_2025

print({len(csv_files)})

{12}


In [ ]:

def clean_logger_data(df, year):
    """
    Should take one raw logger dataframe and returns a clean one.
    df   = the raw table
    year = the year number (2022)
    """

    print(f"  Shape after loading: {df.shape[0]} rows x {df.shape[1]} columns")

    # Select columns by position: 1=Timestamp, 2=DO mg/L, 3=Temp °F
    # Position 0 is the HOBO row counter, which we don't need
    df_clean = df.iloc[:, [1, 2, 3]].copy()

    df_clean.columns = ["Timestamp", "DO_mgL", "Temp_F"]

    df_clean["Year"] = year

    # Parse timestamps; unreadable values become NaN
    df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")

    # Force DO and Temp to numeric in case they loaded as text
    df_clean["DO_mgL"] = pd.to_numeric(df_clean["DO_mgL"], errors="coerce")
    df_clean["Temp_F"] = pd.to_numeric(df_clean["Temp_F"], errors="coerce")

    # Drop rows with unreadable timestamps
    rows_before = len(df_clean)
    df_clean = df_clean.dropna(subset=["Timestamp"])
    rows_after = len(df_clean)
    print(f"  Removed {rows_before - rows_after} rows with bad dates.")

    # Drop rows where both DO and Temp are blank (device maintenance events)
    df_clean = df_clean.dropna(subset=["DO_mgL", "Temp_F"], how="all")

    # Remove physically impossible readings — sensor likely out of water
    rows_before = len(df_clean)
    df_clean = df_clean[
        (df_clean["DO_mgL"]  >  0)   &   # DO must be positive
        (df_clean["DO_mgL"]  <= 20)  &   # DO over 20 mg/L = sensor out of water
        (df_clean["Temp_F"]  >  32)  &   # Must be above freezing
        (df_clean["Temp_F"]  <  95)      # Under 95°F
    ]
    rows_after = len(df_clean)
    print(f"  Fitered out {rows_before - rows_after} out-of-range/bad readings.")
    print(f"   {rows_after} after cleaning.")

    return df_clean

In [ ]:

all_clean_dataframes = []

for file_label, url in csv_files.items():

    print(f"\n {file_label}")

    try:
        # HOBO files have two header rows before the data starts:
        # Row 1: plot title (junk), Row 2: column names (we rename these ourselves)
        df_raw = pd.read_csv(url, skiprows=2, header=None)

        # Extract year from the file label (first 4 characters)
        year = int(file_label[:4])

        df_clean = clean_logger_data(df_raw, year)

        if df_clean is not None and len(df_clean) > 0:
            all_clean_dataframes.append(df_clean)
        else:
            print(f" No valid data returned for {file_label}, being skipped.")

    except Exception as e:
        print(f" ERROR on {file_label}: {e}")



 2022_1
  Shape after loading: 7014 rows x 9 columns


/tmp/ipykernel_227/2384571413.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")


  Removed 0 rows with bad dates.
  Fitered out 0 out-of-range/bad readings.
   7010 after cleaning.

 2022_2
  Shape after loading: 8041 rows x 10 columns


/tmp/ipykernel_227/2384571413.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")


  Removed 0 rows with bad dates.
  Fitered out 96 out-of-range/bad readings.
   7940 after cleaning.

 2023_1
  Shape after loading: 2429 rows x 9 columns


...

  Removed 0 rows with bad dates.
  Fitered out 899 out-of-range/bad readings.
   2470 after cleaning.

 2024_4
  Shape after loading: 13183 rows x 10 columns


/tmp/ipykernel_227/2384571413.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")


  Removed 0 rows with bad dates.
  Fitered out 4518 out-of-range/bad readings.
   8659 after cleaning.

 2025_1
  Shape after loading: 1925 rows x 8 columns
  Removed 0 rows with bad dates.
  Fitered out 0 out-of-range/bad readings.
   1922 after cleaning.

 2025_2


/tmp/ipykernel_227/2384571413.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")
/tmp/ipykernel_227/2384571413.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"], errors="coerce")


  Shape after loading: 7212 rows x 8 columns
  Removed 0 rows with bad dates.
  Fitered out 412 out-of-range/bad readings.
   6797 after cleaning.

 2025_3
  Shape after loading: 3332 rows x 10 columns
  Removed 0 rows with bad dates.
  Fitered out 1660 out-of-range/bad readings.
   1668 after cleaning.


In [ ]:
# Stack all cleaned yearly dataframes into one master table
master_df = pd.concat(all_clean_dataframes, ignore_index=True)

# Sort chronologically
master_df = master_df.sort_values("Timestamp").reset_index(drop=True)


print(f"Total rows:{len(master_df):,}")
print(f"Date range:{master_df['Timestamp'].min()} → {master_df['Timestamp'].max()}")
print(f"Years found:{sorted(master_df['Year'].unique())}")
print(f"Here's a look")
print(master_df.head())

Total rows:53,075
Date range:2022-08-08 16:15:00 → 2025-11-24 20:42:00
Years found:[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Here's a look
            Timestamp  DO_mgL  Temp_F  Year
0 2022-08-08 16:15:00    7.77   78.30  2022
1 2022-08-08 16:25:00    7.69   77.94  2022
2 2022-08-08 16:35:00    7.64   83.66  2022
3 2022-08-08 16:45:00    7.46   86.04  2022
4 2022-08-08 16:55:00    7.42   86.43  2022


In [ ]:
import io

# Estimate output file size before saving
buffer = io.BytesIO()
master_df.to_csv(buffer, index=False)
file_size_bytes = buffer.tell()
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"CSV size: {file_size_mb:.2f} MB?")

CSV size: 1.82 MB?


In [ ]:
output_filename = "Master_Logger_Daily_2022_2025.csv"

In [ ]:
# Save merged dataset to CSV
master_df.to_csv(output_filename, index=False)